# Mock Exam

### Q1. Consider the dataset Sacremento.csv. Ignoring the variable “zip”, consider “price” as the response variable and all other as features. With K-Fold cross-validation with Grid Search with parameters of your choice, evaluate r2 score with following algorithms:
    1. ElasticNet
    2. Random Forest
    3. XG Boost
    4. LightGBM

In [96]:
import numpy as np 
import pandas as pd
from pca import pca
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import silhouette_score
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [34]:
df = pd.read_csv("Datasets/Sacremento.csv")
df.head()

,city,zip,beds,baths,sqft,type,price,latitude,longitude
0,SACRAMENTO,z95838,2,1.0,836,Residential,59222,38.631913,-121.434879
1,SACRAMENTO,z95823,3,1.0,1167,Residential,68212,38.478902,-121.431028
2,SACRAMENTO,z95815,2,1.0,796,Residential,68880,38.618305,-121.443839
3,SACRAMENTO,z95815,2,1.0,852,Residential,69307,38.616835,-121.439146
4,SACRAMENTO,z95824,2,1.0,797,Residential,81900,38.519470,-121.435768


In [35]:
df.isnull().sum()

city         0
zip          0
beds         0
baths        0
sqft         0
type         0
price        0
latitude     0
longitude    0
dtype: int64

In [36]:
X, y = df.drop(["zip","price"], axis=1), df["price"]

In [37]:
print(X.isnull().sum())
print(y.isnull().sum())

city         0
beds         0
baths        0
sqft         0
type         0
latitude     0
longitude    0
dtype: int64
0


In [38]:
ohc = OneHotEncoder(sparse_output=False,handle_unknown="ignore", drop="first").set_output(transform="pandas")
ct = make_column_transformer((ohc,make_column_selector(dtype_include=object)),
                             ("passthrough", make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

### 1.1 Elastic Net

In [39]:
ElasticNet = ElasticNet(random_state=26)
scaler = StandardScaler()
pipe = Pipeline([("CT",ct),("SCL",scaler),("ElasticNet",ElasticNet)])
pipe.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  handle_unknown='ignore',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000276E16C3830>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000276E184FDA0>)],
                     verbose_feature_names_out=False)),
  ('SCL', StandardScaler()),
  ('ElasticNet', ElasticNet(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                handle_unknown='ignore',
            

In [40]:
kfold = KFold(random_state=26,shuffle=True, n_splits=5) 
params = {"ElasticNet__alpha" : np.linspace(0.001,10,20),
          "ElasticNet__l1_ratio" : np.linspace(0.001,1,20)
         }
gcv = GridSearchCV(pipe, param_grid=params,cv=kfold, scoring="r2",n_jobs=-1)
gcv.fit(X,y)
print("Best params : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best params :  {'ElasticNet__alpha': np.float64(0.5272631578947369), 'ElasticNet__l1_ratio': np.float64(0.9474210526315789)}
Best score :  0.6607025330500205


## Random Forest

In [42]:
en = RandomForestRegressor(random_state=26)
pipe = Pipeline([("CT",ct),("scaler",scaler),("rf",en)])
pipe.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  handle_unknown='ignore',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000276E16C3830>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000276E184FDA0>)],
                     verbose_feature_names_out=False)),
  ('scaler', StandardScaler()),
  ('rf', RandomForestRegressor(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                handle_unknown='ignore',
      

In [44]:
params = { 
    "rf__max_features": [2,3,4,5],
    "rf__n_estimators":  [10,25,50,100],
    "rf__max_depth": [None, 5, 10, 15],
    "rf__min_samples_split": [2, 5, 10],
}
kfold = KFold(n_splits=5,shuffle=True, random_state=26)
gcv = GridSearchCV(pipe,param_grid=params,cv =kfold, n_jobs=-1, scoring="r2")
gcv.fit(X,y)
print("Best Params : " , gcv.best_params_)
print("Best Score : " , gcv.best_score_)

Best Params :  {'rf__max_depth': None, 'rf__max_features': 5, 'rf__min_samples_split': 5, 'rf__n_estimators': 100}
Best Score :  0.6715726970229172


## XGBOOST

In [45]:
from xgboost import XGBRegressor

In [46]:

gbm = XGBRegressor(random_state=26)
pipe = Pipeline([("CT",ct),("scaler",scaler),("gbm",gbm)])
pipe.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  handle_unknown='ignore',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000276E16C3830>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000276E184FDA0>)],
                     verbose_feature_names_out=False)),
  ('scaler', StandardScaler()),
  ('gbm',
   XGBRegressor(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=None, early_stopping_rounds=None,
                enable_categorical=True, eval_metric=None, feature_types=None,

In [48]:
params = {
   
    'gbm__learning_rate':  np.linspace(0.01,0.9,10),
    'gbm__max_depth':  [None,2,3,4],
    'gbm__n_estimators':  [25,50,100]
}

kfold = KFold(n_splits=5,shuffle=True, random_state=26)
gcv = GridSearchCV(pipe,param_grid=params,cv =kfold, n_jobs=-1, scoring="r2")
gcv.fit(X,y)
print("Best Params : " , gcv.best_params_)
print("Best Score : " , gcv.best_score_)

Best Params :  {'gbm__learning_rate': np.float64(0.20777777777777778), 'gbm__max_depth': 2, 'gbm__n_estimators': 100}
Best Score :  0.6841161131858826


In [49]:
import lightgbm as lgb
gbm = lgb.LGBMRegressor(random_state=26, verbose=-1)
pipe = Pipeline([("CT",ct),("SCL",scaler),("gbm",gbm)])
pipe.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  handle_unknown='ignore',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000276E16C3830>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000276E184FDA0>)],
                     verbose_feature_names_out=False)),
  ('SCL', StandardScaler()),
  ('gbm', LGBMRegressor(random_state=26, verbose=-1))],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                handle_unknown='ignore',
    

In [50]:
params = {
    'gbm__learning_rate':  np.linspace(0.01,0.9,10),
    'gbm__max_depth':  [None,2,3,4],
    'gbm__n_estimators':  [25,50,100]
}

kfold = KFold(n_splits=5,shuffle=True, random_state=26)
gcv = GridSearchCV(pipe,param_grid=params,cv =kfold, n_jobs=-1, scoring="r2")
gcv.fit(X,y)
print("Best Params : " , gcv.best_params_)
print("Best Score : " , gcv.best_score_)

Best Params :  {'gbm__learning_rate': np.float64(0.20777777777777778), 'gbm__max_depth': 3, 'gbm__n_estimators': 100}
Best Score :  0.6735458470121809


## Q2. 
1. Consider the dataset Protein.xlsx. Find best K, for K-Means clustering on the basis of Silhouette Scores
2. State the number PCAs which provide 80% variation

### 2.1 K-Means clustering

In [58]:
df = pd.read_excel("Datasets/Protein.xlsx",index_col=0)
df.head()

,RedMeat,WhiteMeat,Eggs,Milk,Fish,Cereals,Starch,Nuts,Fr&Veg
Country,,,,,,,,,
Albania,10.1,1.4,0.5,8.9,0.2,42.3,0.6,5.5,1.7
Austria,8.9,14.0,4.3,19.9,2.1,28.0,3.6,1.3,4.3
Belgium,13.5,9.3,4.1,17.5,4.5,26.6,5.7,2.1,4.0
Bulgaria,7.8,6.0,1.6,8.3,1.2,56.7,1.1,3.7,4.2
Czechoslovakia,9.7,11.4,2.8,12.5,2.0,34.3,5.0,1.1,4.0


In [59]:
df.isnull().sum().sum()

np.int64(0)

In [60]:
scaler = StandardScaler().set_output(transform='pandas')
scl_df = scaler.fit_transform(df)

In [61]:
scl_df.head()

,RedMeat,WhiteMeat,Eggs,Milk,Fish,Cereals,Starch,Nuts,Fr&Veg
Country,,,,,,,,,
Albania,0.082941,-1.794750,-2.224584,-1.179570,-1.225033,0.934804,-2.295965,1.247968,-1.378251
Austria,-0.282974,1.686446,1.245621,0.400468,-0.655111,-0.395051,-0.422218,-0.910790,0.092789
Belgium,1.119699,0.387905,1.062979,0.055732,0.064791,-0.525246,0.889405,-0.499598,-0.076947
Bulgaria,-0.618396,-0.523837,-1.220051,-1.265754,-0.925074,2.273959,-1.983674,0.322786,0.036210
Czechoslovakia,-0.039031,0.968104,-0.124197,-0.662467,-0.685107,0.190830,0.452198,-1.013588,-0.076947


In [72]:
ks = np.arange(2,21)
ss = []
for k in ks:
    km = KMeans(n_clusters=k,random_state=26)
    km.fit(scl_df)
    score = silhouette_score(scl_df,km.labels_)
    ss.append([k,score])
df_ss = pd.DataFrame(ss,columns=["K","Silhouette_Score"])
df_ss.sort_values("Silhouette_Score",ascending=False)

,K,Silhouette_Score
0,2,0.327108
3,5,0.263945
4,6,0.252868
5,7,0.226217
6,8,0.217537
1,3,0.211785
2,4,0.198055
8,10,0.191925
10,12,0.190520
9,11,0.187782


## 2.2 PCA
- State the number PCAs which provide 80% variation

In [ ]:
scaler = StandardScaler().set_output(transform='pandas')
pca = PCA()
pipe = Pipeline([("SCL",scaler),("PCA",pca),("LR",lr)])

In [83]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.80)
pca.fit(scl_df)

print("Number of PCs for 80% variation:", pca.n_components_)

Number of PCs for 80% variation: 4


## Q3. Consider the dataset Vehicle.csv. Consider the variable “Class” as response variable. 
1. With Grid Search for Decision Tree considering parameter values for max_depth as [2,4,6] , min_samples_split  as [2,3,5] and min_samples_leaf as [1,3,5], find best parameter set on the basis of log loss.
2. With Grid Search for Random Forest considering parameter values for max_features=[2,3,4,5]
3. With Grid Search for Logistic Regression considering parameter values of solvers

#### 3.1 With Grid Search for Decision Tree considering parameter values for max_depth as [2,4,6] , min_samples_split  as [2,3,5] and min_samples_leaf as [1,3,5], find best parameter set on the basis of log loss.

In [85]:
df = pd.read_csv("Day05/Vehicle.csv")
df.head()

,Comp,Circ,D.Circ,Rad.Ra,Pr.Axis.Ra,Max.L.Ra,Scat.Ra,Elong,Pr.Axis.Rect,Max.L.Rect,Sc.Var.Maxis,Sc.Var.maxis,Ra.Gyr,Skew.Maxis,Skew.maxis,Kurt.maxis,Kurt.Maxis,Holl.Ra,Class
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus


In [88]:
X, y = df.drop("Class", axis=1), df["Class"]

In [89]:
scaler = StandardScaler().set_output(transform='pandas')

In [91]:
dtc = DecisionTreeClassifier(random_state=26)
pipe = Pipeline([("scaler",scaler),("dtc",dtc)])
pipe.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('dtc', DecisionTreeClassifier(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'dtc': DecisionTreeClassifier(random_state=26),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'dtc__ccp_alpha': 0.0,
 'dtc__class_weight': None,
 'dtc__criterion': 'gini',
 'dtc__max_depth': None,
 'dtc__max_features': None,
 'dtc__max_leaf_nodes': None,
 'dtc__min_impurity_decrease': 0.0,
 'dtc__min_samples_leaf': 1,
 'dtc__min_samples_split': 2,
 'dtc__min_weight_fraction_leaf': 0.0,
 'dtc__monotonic_cst': None,
 'dtc__random_state': 26,
 'dtc__splitter': 'best'}

In [94]:
params = {
    'dtc__max_depth':  [2,4,6],
    'dtc__min_samples_split':  [2,3,5],
    'dtc__max_leaf_nodes':  [1,3,5]
}
kfold = KFold(n_splits=5,shuffle=True, random_state=26)
gcv = GridSearchCV(pipe,param_grid=params,cv =kfold, n_jobs=-1, scoring="accuracy")
gcv.fit(X,y)
print("Best Params : " , gcv.best_params_)
print("Best Score : " , gcv.best_score_)

Best Params :  {'dtc__max_depth': 4, 'dtc__max_leaf_nodes': 5, 'dtc__min_samples_split': 2}
Best Score :  0.6158510268012531


C:\Users\PGCP-AI.STUDENTSDC\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_validation.py:493: FitFailedWarning: 
45 fits failed out of a total of 135.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
45 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\PGCP-AI.STUDENTSDC\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\PGCP-AI.STUDENTSDC\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PGCP-

#### 3.2 With Grid Search for Random Forest considering parameter values for max_features=[2,3,4,5]

In [ ]:
RF = RandomForestClassifier(random_state=26)
pipe = Pipeline([("scaler",scaler),("RF",RF)])
pipe.get_params()